In [2]:
import os
import json
import random
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn import svm
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import GridSearchCV
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.under_sampling import RandomUnderSampler, NearMiss
from imblearn.over_sampling import SMOTE, RandomOverSampler
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.layers import GRU
from sklearn.feature_extraction.text import TfidfVectorizer

# Set random seed for NumPy
np.random.seed(42)

# Set random seed for TensorFlow v1
tf.random.set_seed(42)

file_path = 'domain1_train_data.json'
raw_data_domain1 = pd.read_json(file_path, lines=True)
file_path = 'domain2_train_data.json'
train_data_domain2 = pd.read_json(file_path, lines=True)
file_path = 'test_data.json'
test_data = pd.read_json(file_path, lines=True)
# split into train and validation data
label_column = 'label'
train_data_domain1, validation_data_domain1 = train_test_split(raw_data_domain1, test_size=0.1, random_state=42, stratify=raw_data_domain1[label_column])
train_domain1_y = np.array(train_data_domain1[label_column])

## extract train_domain1_x
vectorizer_domain1 = TfidfVectorizer()

# Fit the vectorizer to the text data
vectorizer_domain1.fit(train_data_domain1['text'].apply(str))

vectorizer_domain2 = TfidfVectorizer()

# Fit the vectorizer to the text data
vectorizer_domain2.fit(train_data_domain2['text'].apply(str))

# Transform the text data into a 2D array
train_domain1_x = vectorizer_domain1.transform(train_data_domain1['text'].apply(str)).toarray()

validation_domain1_y = np.array(validation_data_domain1[label_column])
validation_domain1_x = vectorizer_domain1.transform(validation_data_domain1['text'].apply(str)).toarray()

train_domain2_y = np.array(train_data_domain2[label_column])
train_domain2_x = vectorizer_domain2.transform(train_data_domain2['text'].apply(str)).toarray()

# Calculate the number of features
num_features = train_domain1_x.shape[1]

"\nearly_stopping = callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)\n\n# Define your model using TensorFlow/Keras\nmodel = models.Sequential([\n    layers.Dense(128, activation='relu', input_shape=(num_features,), kernel_regularizer=regularizers.l2(0.01)),\n    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.01)),\n    layers.Dropout(0.5),\n    layers.Dense(1, activation='sigmoid')\n])\n\nmodel_2 = models.Sequential([\n    layers.Dense(256, activation='relu', input_shape=(num_features,), kernel_regularizer=regularizers.l2(0.01)),\n    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.01)),\n    layers.Dropout(0.5),\n    layers.Dense(1, activation='sigmoid')\n])\n\noptimizer = tf.keras.optimizers.Adam(learning_rate=0.002)\noptimizer_2 = tf.keras.optimizers.Adam(learning_rate=0.002)\n\n# Compile the model\nmodel.compile(optimizer=optimizer,\n              loss='binary_crossentropy',\n              me

In [3]:
# Instantiate Random Forest Classifier for domain 1 data
rf_classifier_1 = RandomForestClassifier(
    bootstrap=False,
    max_depth=None,
    max_features='sqrt',
    min_samples_leaf=1,
    min_samples_split=5,
    n_estimators=200,
    random_state=42
)

# Train the model
rf_classifier_1.fit(train_domain1_x, train_domain1_y)

# Predict the labels of the validation set
validation_predictions_rf = rf_classifier_1.predict(validation_domain1_x)

# Calculate the precision, recall, and F1 score of the model
precision_rf = precision_score(validation_domain1_y, validation_predictions_rf)
recall_rf = recall_score(validation_domain1_y, validation_predictions_rf)
f1_rf = f1_score(validation_domain1_y, validation_predictions_rf)

# Print the precision, recall, and F1 score of the model
print(f'Precision: {precision_rf}')
print(f'Recall: {recall_rf}')
print(f'F1 Score: {f1_rf}')

Precision: 0.7582417582417582
Recall: 0.828
F1 Score: 0.7915869980879542


In [4]:
# Train a list of model for domain 2 data
# Separate minority and majority class instances
minority_class = train_data_domain2[train_data_domain2[label_column] == 1]
majority_class = train_data_domain2[train_data_domain2[label_column] == 0]

# Shuffle the majority class instances
majority_class = majority_class.sample(frac=1, random_state=42)

# Calculate the number of subsets (models) based on the ratio of majority to minority class instances
num_subsets = len(majority_class) // len(minority_class)

# Initialize a list to hold the trained models
models_rf_domain_2 = []

# Loop through each subset, pair minority class with a portion of the majority class, and train a model
for i in range(num_subsets):
    # Select a subset of the majority class
    subset_majority = majority_class[i * len(minority_class):(i + 1) * len(minority_class)]
    
    # Combine minority and majority class instances
    subset_data = pd.concat([minority_class, subset_majority])

    # Shuffle the subset data
    subset_data = subset_data.sample(frac=1, random_state=42)
    
    subset_x, validation_domain2_x, subset_y, validation_domain2_y = train_test_split(subset_data['text'], subset_data[label_column], test_size=0.1, random_state=42, stratify=subset_data[label_column])
    
    # Split data into features and labels
    subset_x = vectorizer_domain2.transform(subset_x.apply(str)).toarray()
    validation_domain2_x = vectorizer_domain2.transform(validation_domain2_x.apply(str)).toarray()
    
    # Train a classifier (Random Forest in this case)
    rf_classifier = RandomForestClassifier(
        bootstrap=False,
        max_depth=None,
        max_features='sqrt',
        min_samples_leaf=1,
        min_samples_split=5,
        n_estimators=200,
        random_state=42
    )

    rf_classifier.fit(subset_x, subset_y)

    validation_predictions_rf = rf_classifier.predict(validation_domain2_x)

    # Calculate the precision, recall, and F1 score of the model
    precision_rf = precision_score(validation_domain2_y, validation_predictions_rf)
    recall_rf = recall_score(validation_domain2_y, validation_predictions_rf)
    f1_rf = f1_score(validation_domain2_y, validation_predictions_rf)

    # Print the precision, recall, and F1 score of the model
    print(f'Precision: {precision_rf}')
    print(f'Recall: {recall_rf}')
    print(f'F1 Score: {f1_rf}')

    print("1", np.sum(validation_predictions_rf == 1))
    print("0", np.sum(validation_predictions_rf == 0))
    
    # Append trained model to the list
    models_rf_domain_2.append(rf_classifier)


Precision: 0.8
Recall: 0.9333333333333333
F1 Score: 0.8615384615384616
1 175
0 125
Precision: 0.8214285714285714
Recall: 0.92
F1 Score: 0.8679245283018867
1 168
0 132
Precision: 0.7932960893854749
Recall: 0.9466666666666667
F1 Score: 0.8632218844984803
1 179
0 121
Precision: 0.8092485549132948
Recall: 0.9333333333333333
F1 Score: 0.86687306501548
1 173
0 127
Precision: 0.8214285714285714
Recall: 0.92
F1 Score: 0.8679245283018867
1 168
0 132
Precision: 0.7777777777777778
Recall: 0.9333333333333333
F1 Score: 0.8484848484848485
1 180
0 120
Precision: 0.7932960893854749
Recall: 0.9466666666666667
F1 Score: 0.8632218844984803
1 179
0 121


In [10]:
# train an binary classifier to split domain1 and domain2
domain1_y = np.zeros(raw_data_domain1.shape[0])
# split domain2 into two subset same size as domain1
# Calculate the size of each subset
subset_size = raw_data_domain1.shape[0]

# Split domain2 into two equal parts
domain2_part1 = train_data_domain2.iloc[:subset_size]
domain2_part2 = train_data_domain2.iloc[subset_size:]

# Sample the same number of samples from each part
domain2_subset1 = domain2_part1.sample(n=subset_size, random_state=42)
domain2_subset2 = domain2_part2.sample(n=subset_size, random_state=42)

domain2_y = np.ones(domain2_subset1.shape[0])
train_domain2_x = vectorizer_domain1.transform(domain2_subset1['text'].apply(str)).toarray()
domain_y = np.concatenate((domain1_y, domain2_y), axis=0)
domain_x = np.concatenate((vectorizer_domain1.transform(raw_data_domain1['text'].apply(str)).toarray(), train_domain2_x), axis=0)

early_stopping = callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)

domain_model = models.Sequential([
    layers.Dense(512, activation='relu', input_shape=(num_features,), kernel_regularizer=regularizers.l2(0.01)),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

domain_model.compile(optimizer='adam',
                loss='binary_crossentropy',
                metrics=['accuracy'])

train_domain_x, validation_domain_x, train_domain_y, validation_domain_y = train_test_split(domain_x, domain_y, test_size=0.1, random_state=42, stratify=domain_y)

checkpoint_filepath_domain = 'best_model_domain.keras'

model_checkpoint_callback_domain = ModelCheckpoint(
    filepath=checkpoint_filepath_domain,
    save_best_only=True,
    monitor='val_accuracy',
    mode='max',
    verbose=1
)

# Train the second model with the new EarlyStopping callback
history_2 = domain_model.fit(train_domain_x, train_domain_y, epochs=50, batch_size=256, validation_data=(validation_domain_x,validation_domain_y), callbacks=[model_checkpoint_callback_domain, early_stopping])

# Load the best model saved during training for the second model
best_model_domain = models.load_model(checkpoint_filepath_domain)

domain2_y = np.ones(domain2_subset2.shape[0])
train_domain2_x = vectorizer_domain1.transform(domain2_subset2['text'].apply(str)).toarray()
domain_y = np.concatenate((domain1_y, domain2_y), axis=0)
domain_x = np.concatenate((vectorizer_domain1.transform(raw_data_domain1['text'].apply(str)).toarray(), train_domain2_x), axis=0)

domain_model = models.Sequential([
    layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

domain_model.compile(optimizer='adam',
                loss='binary_crossentropy',
                metrics=['accuracy'])

train_domain_x, validation_domain_x, train_domain_y, validation_domain_y = train_test_split(domain_x, domain_y, test_size=0.1, random_state=42, stratify=domain_y)

checkpoint_filepath_domain = 'best_model_domain_2.keras'

model_checkpoint_callback_domain = ModelCheckpoint(
    filepath=checkpoint_filepath_domain,
    save_best_only=True,
    monitor='val_accuracy',
    mode='max',
    verbose=1
)

# Train the second model with the new EarlyStopping callback
history_2 = domain_model.fit(train_domain_x, train_domain_y, epochs=50, batch_size=256, validation_data=(validation_domain_x,validation_domain_y), callbacks=[model_checkpoint_callback_domain, early_stopping])

# Load the best model saved during training for the second model
best_model_domain2 = models.load_model(checkpoint_filepath_domain)

d:\Anaconda\envs\NLP\Lib\site-packages\keras\src\layers\core\dense.py:86: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 0.9254 - loss: 4.2537
Epoch 1: val_accuracy improved from -inf to 1.00000, saving model to best_model_domain.keras
36/36 ━━━━━━━━━━━━━━━━━━━━ 5s 111ms/step - accuracy: 0.9268 - loss: 4.1947 - val_accuracy: 1.0000 - val_loss: 0.5243
Epoch 2/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.9942 - loss: 0.4654
Epoch 2: val_accuracy did not improve from 1.00000
36/36 ━━━━━━━━━━━━━━━━━━━━ 4s 97ms/step - accuracy: 0.9942 - loss: 0.4644 - val_accuracy: 1.0000 - val_loss: 0.3705
Epoch 3/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - accuracy: 0.9938 - loss: 0.3624
Epoch 3: val_accuracy did not improve from 1.00000
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 96ms/step - accuracy: 0.9939 - loss: 0.3620 - val_accuracy: 1.0000 - val_loss: 0.3230
Epoch 4/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.9945 - loss: 0.3209
Epoch 4: val_accuracy did not improve from 1.00000
36/36 ━━━━━━━━━━━━━━━━━━━━ 4s 97ms/step - accuracy: 0.9945 - loss

In [6]:
# split the test data into domain1 and domain2
test_x = vectorizer_domain1.transform(test_data['text'].apply(str)).toarray()
domain_pred = best_model_domain.predict(test_x)
test_x = vectorizer_domain1.transform(test_data['text'].apply(str)).toarray()
domain_pred_1 = best_model_domain2.predict(test_x)
domain_pred = np.mean([domain_pred, domain_pred_1], axis=0)
domain_pred = np.where(domain_pred > 0.5, 1, 0)
domain1_index = np.where(domain_pred == 0)
domain2_index = np.where(domain_pred == 1)
test_domain_1 = test_data[domain_pred == 0]
test_domain_2 = test_data[domain_pred == 1]
print('domain1:', len(test_domain_1))
print('domain2:', len(test_domain_2))

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
domain1: 1987
domain2: 2013


In [8]:
# get the prediction for domain1
test_predict_domain1 = rf_classifier_1.predict(vectorizer_domain1.transform(test_domain_1['text'].apply(str)).toarray())

predictions_per_model_domain2 = []
for rf_model in models_rf_domain_2:
    predictions = rf_model.predict(vectorizer_domain2.transform(test_domain_2['text'].apply(str)).toarray())
    predictions_per_model_domain2.append(predictions)

# Calculate the average of the predictions
average_predictions_domain2 = np.mean(predictions_per_model_domain2, axis=0)
average_predictions_domain2 = np.where(average_predictions_domain2 > 0.99, 1, 0)

test_predict_domain2 = average_predictions_domain2

test_predict_domain1 = np.where(test_predict_domain1 > 0.5, 1, 0)
print('1:', np.sum(test_predict_domain1 == 1))
print('0:', np.sum(test_predict_domain1 == 0))
test_predict_domain2 = np.where(test_predict_domain2 > 0.5, 1, 0)
print('1:', np.sum(test_predict_domain2 == 1))
print('0:', np.sum(test_predict_domain2 == 0))

1: 1047
0: 940
1: 977
0: 1036


In [9]:
# Initialize the 'label' column with default values
test_data['label'] = 0

# Update the 'label' column with predictions for domain 1
test_data.loc[domain1_index[0], 'label'] = test_predict_domain1.flatten()

# Update the 'label' column with predictions for domain 2
test_data.loc[domain2_index[0], 'label'] = test_predict_domain2.flatten()

# Convert the 'label' column to integer type
test_data['label'] = test_data['label'].astype(int)

print('1:', np.sum(test_data['label'] == 1))
print('0:', np.sum(test_data['label'] == 0))


# Save the predictions to a CSV file
test_data[['id', 'label']].to_csv('sample.csv', index=False)

1: 2024
0: 1976
